# Prediction MASI pour mai et juin

Objectif: produire une prediction journaliere + un resume mensuel pour mai et juin 2026, et exporter un rapport markdown.

Le notebook s'appuie sur le module `masi_hybrid_forecasting.pipeline.forecast` :

1. Si le fichier canonique `outputs/etape6/etape6_final_predictions.csv` couvre la fenetre cible (cas ou les etapes 01-06 ont ete relancees sur des donnees fraiches), ses predictions sont reutilisees.
2. Sinon, projection hors-echantillon par ARIMA (selection AIC sur 8 ordres) + simulation Monte-Carlo (5000 trajectoires) pour des bandes p10/p50/p90 robustes.
3. Conditionnement par le dernier regime HMM : si Bear/Bull, on decale la derive par le rendement moyen historique du regime. Si Neutral, derive nulle.

Tous les artefacts sont ecrits dans `outputs/etape10/`. La meme logique est disponible en CLI :

```bash
python -m masi_hybrid_forecasting.pipeline forecast --year 2026 --months 5,6 --n-mc 5000
```

## 1. Imports et chemins

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "masi_hybrid_forecasting").exists():
            return candidate
    raise RuntimeError("Racine du projet introuvable. Lance le notebook depuis le depot.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from masi_hybrid_forecasting.pipeline import forecast as fc_mod
from masi_hybrid_forecasting.pipeline.config import CANONICAL_CSV

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "etape10"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Projet  : {PROJECT_ROOT}")
print(f"Module  : {fc_mod.__file__}")
print(f"Sortie  : {OUTPUT_DIR}")

## 2. Parametres

Modifie `TARGET_YEAR` / `TARGET_MONTHS` pour cibler une autre fenetre. `N_MC` controle la finesse des bandes Monte-Carlo.

In [ ]:
TARGET_YEAR = 2026
TARGET_MONTHS = [5, 6]
N_MC = 5_000
USE_REGIME_DRIFT = True
SEED = 1234

TARGET_START = pd.Timestamp(year=TARGET_YEAR, month=min(TARGET_MONTHS), day=1)
TARGET_END = pd.Timestamp(year=TARGET_YEAR, month=max(TARGET_MONTHS), day=1) + pd.offsets.MonthEnd(0)

OUTPUT_DAILY_CSV = OUTPUT_DIR / f"forecast_may_june_{TARGET_YEAR}_daily.csv"
OUTPUT_MONTHLY_CSV = OUTPUT_DIR / f"forecast_may_june_{TARGET_YEAR}_monthly.csv"
OUTPUT_REPORT_MD = OUTPUT_DIR / "report.md"
OUTPUT_FIG_FAN = OUTPUT_DIR / f"forecast_{TARGET_YEAR}_fan.png"
OUTPUT_FIG_MONTHLY = OUTPUT_DIR / f"forecast_{TARGET_YEAR}_monthly_distribution.png"

print(f"Periode cible : {TARGET_START.date()} -> {TARGET_END.date()}")
print(f"Monte-Carlo   : {N_MC:,} trajectoires (seed={SEED})")
print(f"Drift regime  : {'on' if USE_REGIME_DRIFT else 'off'}")

## 3. Historique MASI et derniere observation

In [ ]:
hist = fc_mod.load_history()
PRICE_COL = fc_mod.PRICE_COL
last_date = hist.index.max()
last_price = float(hist[PRICE_COL].iloc[-1])

print(f"Historique     : {hist.index.min().date()} -> {last_date.date()} | {len(hist):,} obs")
print(f"Dernier close  : {last_price:,.2f}")
print(f"Volatilite 21j : {hist['log_return'].tail(21).std() * np.sqrt(252):.2%} annualisee")
hist.tail()

## 4. Couverture pipeline pour mai/juin (si dispo)

Si tu as relance `python -m masi_hybrid_forecasting.pipeline train` / `predict` sur des donnees mises a jour, ces lignes seront utilisees directement par `forecast` (source = `pipeline`).

In [ ]:
pipeline_window = fc_mod.load_pipeline_window(TARGET_YEAR, TARGET_MONTHS)

if pipeline_window.empty:
    print(f"Aucune prediction pipeline pour {TARGET_YEAR}-{TARGET_MONTHS}.")
    print(f"-> Projection ARIMA + Monte-Carlo seront utilisees.")
else:
    print(f"Pipeline couvre {len(pipeline_window)} jours dans la fenetre cible.")
    pipeline_window["month"] = pipeline_window["date"].dt.to_period("M").astype(str)
    display(
        pipeline_window.groupby("month")
        .agg(
            n_days=("date", "size"),
            mean_predicted_return=("predicted_return", "mean"),
            positive_days=("predicted_return", lambda s: int((s > 0).sum())),
            negative_days=("predicted_return", lambda s: int((s < 0).sum())),
        )
    )

## 5. Lancement de la projection hybride

Une seule cellule fait tout : ARIMA + Monte-Carlo + drift regime + concatenation observed/pipeline/forecast. Resultat dans `ForecastResult`.

In [ ]:
result = fc_mod.run_forecast(
    target_year=TARGET_YEAR,
    target_months=TARGET_MONTHS,
    n_mc=N_MC,
    use_regime_drift=USE_REGIME_DRIFT,
    seed=SEED,
)

print(f"ARIMA retenu       : {result.arima_order} | AIC = {result.arima_aic:,.2f}")
print(f"Drift regime HMM   : {result.regime_drift:+.5f} log-return/jour")
print(f"Trajectoires MC    : {result.n_mc:,}")
print(f"Dernier obs        : {result.last_obs_date.date()} (close {result.last_obs_close:,.2f})")
print(f"Jours dans le panel : {len(result.daily)} (mai+juin {TARGET_YEAR})")
result.daily.head()

## 6. Tableau journalier mai-juin (export CSV)

In [ ]:
daily = result.daily.copy()
daily.to_csv(OUTPUT_DAILY_CSV, index=False)
print(f"CSV journalier ecrit : {OUTPUT_DAILY_CSV}")
print(f"Sources presentes    : {sorted(daily['source'].unique())}")
print(f"Signaux               : {daily['signal'].value_counts().to_dict()}")
daily.head(10)

## 7. Resume mensuel (export CSV + rapport markdown)

In [ ]:
monthly = result.monthly.copy()
monthly.to_csv(OUTPUT_MONTHLY_CSV, index=False)
fc_mod.write_report(result, OUTPUT_REPORT_MD, TARGET_YEAR)
print(f"CSV mensuel ecrit  : {OUTPUT_MONTHLY_CSV}")
print(f"Rapport markdown   : {OUTPUT_REPORT_MD}")
monthly

## 8. Fan chart : prix + bandes Monte-Carlo p10/p50/p90

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))

# 1. Historique recent (12 mois glissants)
lookback_start = last_date - pd.Timedelta(days=365)
recent_hist = hist.loc[hist.index >= lookback_start, PRICE_COL]
ax.plot(recent_hist.index, recent_hist.values,
        label="historique MASI", color="#111827", linewidth=1.6)

# 2. Forecast + bandes
fc_rows = daily[daily["source"] == "forecast"].dropna(subset=["predicted_close"]).copy()
if not fc_rows.empty and "mc_p10" in fc_rows.columns:
    ax.plot(fc_rows["date"], fc_rows["mc_p50"],
            label="projection p50 (Monte-Carlo)", color="#2563eb", linewidth=2.0)
    ax.fill_between(fc_rows["date"], fc_rows["mc_p10"], fc_rows["mc_p90"],
                    color="#93c5fd", alpha=0.35, label="bande p10-p90 (MC)")
elif not fc_rows.empty:
    ax.plot(fc_rows["date"], fc_rows["predicted_close"],
            label="projection ARIMA", color="#2563eb", linewidth=2.0)

# 3. Observed dans la fenetre (si la fenetre est deja couverte)
obs_rows = daily[daily["source"] == "observed"]
if not obs_rows.empty:
    ax.scatter(obs_rows["date"], obs_rows["predicted_close"],
               s=18, color="#dc2626", label="observed (deja realise)", zorder=5)

ax.axvspan(TARGET_START, TARGET_END, color="#f59e0b", alpha=0.12, label="mai-juin cible")
ax.set_title(f"Projection MASI mai-juin {TARGET_YEAR} | ARIMA {result.arima_order} + {result.n_mc:,} MC")
ax.set_ylabel("Niveau MASI")
ax.legend(loc="best", fontsize=9)
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(OUTPUT_FIG_FAN, dpi=110)
plt.show()
print(f"Figure enregistree : {OUTPUT_FIG_FAN}")

## 9. Distribution Monte-Carlo des rendements mensuels

On simule les trajectoires de prix et on calcule le rendement cumule par mois pour visualiser la dispersion.

In [ ]:
if result.arima_order is None:
    print("Periode cible deja couverte par l'historique : pas de Monte-Carlo a simuler.")
else:
    from statsmodels.tsa.arima.model import ARIMA

    returns = hist["log_return"]
    fit = ARIMA(returns, order=result.arima_order, trend="c",
                enforce_stationarity=False, enforce_invertibility=False).fit()
    forecast_dates = pd.bdate_range(last_date + pd.offsets.BDay(1), TARGET_END)
    fc_returns = np.asarray(fit.get_forecast(steps=len(forecast_dates)).predicted_mean, dtype=float)
    sigma = float(np.nanstd(fit.resid, ddof=1))

    paths = fc_mod.simulate_paths(
        mean_returns=fc_returns, sigma=sigma, n_sim=N_MC,
        horizon=len(forecast_dates), last_price=last_price,
        drift_shift=result.regime_drift, seed=SEED,
    )

    panel = pd.DataFrame(paths.T, index=forecast_dates)
    panel.columns = [f"sim_{i}" for i in range(N_MC)]

    monthly_returns_mc = {}
    prior_close = last_price
    for m in TARGET_MONTHS:
        m_start = pd.Timestamp(year=TARGET_YEAR, month=m, day=1)
        m_end = m_start + pd.offsets.MonthEnd(0)
        sub = panel.loc[(panel.index >= m_start) & (panel.index <= m_end)]
        if sub.empty:
            continue
        ret_pct = 100 * (sub.iloc[-1].values / prior_close - 1)
        monthly_returns_mc[m_start.strftime("%Y-%m")] = ret_pct
        prior_close = float(np.median(sub.iloc[-1].values))

    fig, axes = plt.subplots(1, len(monthly_returns_mc), figsize=(6 * len(monthly_returns_mc), 4.2),
                              sharey=True, squeeze=False)
    for ax, (month, vals) in zip(axes[0], monthly_returns_mc.items()):
        ax.hist(vals, bins=60, color="#3b82f6", alpha=0.75, edgecolor="white")
        med = float(np.median(vals))
        ax.axvline(med, color="#1e3a8a", linewidth=1.5, linestyle="--",
                   label=f"mediane {med:+.2f} %")
        ax.axvline(np.percentile(vals, 10), color="#ef4444", linewidth=1.0, linestyle=":",
                   label=f"p10 {np.percentile(vals, 10):+.2f} %")
        ax.axvline(np.percentile(vals, 90), color="#16a34a", linewidth=1.0, linestyle=":",
                   label=f"p90 {np.percentile(vals, 90):+.2f} %")
        ax.set_title(f"Rendement {month} - {N_MC:,} sims")
        ax.set_xlabel("Rendement mensuel (%)")
        ax.legend(fontsize=8)
    axes[0, 0].set_ylabel("Frequence")
    plt.tight_layout()
    plt.savefig(OUTPUT_FIG_MONTHLY, dpi=110)
    plt.show()
    print(f"Figure enregistree : {OUTPUT_FIG_MONTHLY}")

    print("\nResume Monte-Carlo (rendements mensuels) :")
    for month, vals in monthly_returns_mc.items():
        print(
            f"  {month} : "
            f"mediane {np.median(vals):+.2f} % | "
            f"p10 {np.percentile(vals, 10):+.2f} % | "
            f"p90 {np.percentile(vals, 90):+.2f} % | "
            f"P(>0) = {100 * (vals > 0).mean():.1f} %"
        )

## 10. Lecture rapide

Synthese textuelle des resultats pour la note memoire.

In [ ]:
if monthly.empty:
    print("Aucun jour disponible dans la periode cible.")
else:
    print(f"== Projection MASI {TARGET_YEAR} (mai + juin) ==")
    print(f"   Dernier observed : {result.last_obs_date.date()} (close {result.last_obs_close:,.2f})")
    print(f"   Modele           : ARIMA {result.arima_order} (AIC {result.arima_aic:,.2f})")
    print(f"   Drift HMM        : {result.regime_drift:+.5f} log-return/jour")
    print(f"   Monte-Carlo      : {result.n_mc:,} trajectoires")
    print()
    for row in monthly.to_dict("records"):
        direction = "hausse" if row["monthly_log_return"] > 0 else "baisse" if row["monthly_log_return"] < 0 else "neutre"
        print(
            f"   {row['month']} | {direction:>6} | "
            f"close fin {row['end_predicted_close']:>9,.2f} | "
            f"rendement {row['monthly_simple_return_pct']:+.2f} % | "
            f"jours+/-: {row['positive_days']}/{row['negative_days']}"
        )

print("\nNote : projection statistique. Pour une prediction hybride complete avec donnees reelles,")
print("       relance les etapes 01 -> 06 du pipeline puis ce notebook.")